En Telecomunicaciones es muy común que exista un KPI, llamado churn que sin tecnicismos brinda una tasa de abandono de clientes o suscriptores durante un tiempo. Este notebook maneja un supuesto donde se mide a los clientes por planes (plus, basic, premium) y ejemplifica lo fácil que es calcularno en python

<p align="center">
  <img src="images/churn.png" alt="churn" width="500"><br>
  <em>así de fácil se calcula el churn</em>
</p>

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. cargar los datasets, primero los clientes
customers = pd.read_excel(r'D:\I\Documents\BDs\Churn\customers.xlsx')
customers.head()

,customer_id,signup_date,country,plan_type,monthly_fee
0,C10001,2023-01-14,MX,Plus,149
1,C10002,2022-11-03,CO,Basic,79
2,C10003,2023-04-22,MX,Premium,199
3,C10004,2023-02-10,AR,Plus,149
4,C10005,2023-03-05,MX,Basic,79


In [3]:
# posteriormente el uso que han tenido los clientes del producto, número de sesiones, cuantos minutos pasó conectado,
# las interacciones que tuvo, etc
usage = pd.read_excel(r'D:\I\Documents\BDs\Churn\logs.xlsx')
usage.head()

,customer_id,month,sessions,minutes,actions
0,C10001,2023-06,10,480,15
1,C10001,2023-07,12,540,18
2,C10001,2023-08,15,610,22
3,C10002,2023-06,4,72,3
4,C10002,2023-07,3,38,2


In [4]:
# 2. calcular usage_score mensual, en este ejemplo = 0.4 * sessions + 0.3 * minutes + 0.3 * actions
usage['usage_score'] = (
    0.4 * usage['sessions']
    + 0.3 * usage['minutes']
    + 0.3 * usage['actions']
)

usage.head()

,customer_id,month,sessions,minutes,actions,usage_score
0,C10001,2023-06,10,480,15,152.5
1,C10001,2023-07,12,540,18,172.2
2,C10001,2023-08,15,610,22,195.6
3,C10002,2023-06,4,72,3,24.1
4,C10002,2023-07,3,38,2,13.2


In [5]:
# 3. obtener los últimos 3 meses por usuario, aseguramos orden por customer_id y month
usage_sorted = usage.sort_values(['customer_id', 'month']).copy()
usage_sorted.head()

,customer_id,month,sessions,minutes,actions,usage_score
0,C10001,2023-06,10,480,15,152.5
1,C10001,2023-07,12,540,18,172.2
2,C10001,2023-08,15,610,22,195.6
3,C10002,2023-06,4,72,3,24.1
4,C10002,2023-07,3,38,2,13.2


In [6]:
# tomamos las últimas 3 filas por cliente
last_3_months = (
    usage_sorted
    .groupby('customer_id', group_keys=True)
    .apply(lambda g: g.tail(3))
).reset_index().drop(columns='level_1')

last_3_months.head()

,customer_id,month,sessions,minutes,actions,usage_score
0,C10001,2023-06,10,480,15,152.5
1,C10001,2023-07,12,540,18,172.2
2,C10001,2023-08,15,610,22,195.6
3,C10002,2023-06,4,72,3,24.1
4,C10002,2023-07,3,38,2,13.2


In [7]:
# 4. calcular avg_monthly_usage_score
avg_usage = (
    last_3_months
    .groupby('customer_id', as_index=False)['usage_score']
    .mean()
    .rename(columns={'usage_score': 'avg_monthly_usage_score'})
)

avg_usage

,customer_id,avg_monthly_usage_score
0,C10001,173.43
1,C10002,13.17
2,C10003,304.70
3,C10004,123.30
4,C10005,65.80
5,C10006,253.90
6,C10007,151.50
7,C10008,8.43
8,C10009,224.70
9,C10010,105.00


In [8]:
# 5. obtener usage_score del último mes
last_month = (
    usage_sorted
    .groupby('customer_id', as_index=False)
    .tail(1)
    [['customer_id', 'usage_score']]
    .rename(columns={'usage_score': 'last_usage_score'})
)

last_month

,customer_id,last_usage_score
2,C10001,195.6
5,C10002,2.2
8,C10003,325.3
11,C10004,110.5
14,C10005,57.1
17,C10006,247.2
20,C10007,152.5
23,C10008,2.8
26,C10009,214.9
29,C10010,114.9


In [9]:
# 6. unir métricas de uso con customers
df = customers.merge(avg_usage, on='customer_id', how='left')
df = df.merge(last_month, on='customer_id', how='left')
df.head()

,customer_id,signup_date,country,plan_type,monthly_fee,avg_monthly_usage_score,last_usage_score
0,C10001,2023-01-14,MX,Plus,149,173.43,195.6
1,C10002,2022-11-03,CO,Basic,79,13.17,2.2
2,C10003,2023-04-22,MX,Premium,199,304.70,325.3
3,C10004,2023-02-10,AR,Plus,149,123.30,110.5
4,C10005,2023-03-05,MX,Basic,79,65.80,57.1


In [10]:
# 7. Calcular CLV (Customer Lifetime Value), es el ingreso que un cliente genera
df['CLV'] = df['monthly_fee'] * df['avg_monthly_usage_score']
df.head()

,customer_id,signup_date,country,plan_type,monthly_fee,avg_monthly_usage_score,last_usage_score,CLV
0,C10001,2023-01-14,MX,Plus,149,173.43,195.6,25841.57
1,C10002,2022-11-03,CO,Basic,79,13.17,2.2,1040.17
2,C10003,2023-04-22,MX,Premium,199,304.70,325.3,60635.30
3,C10004,2023-02-10,AR,Plus,149,123.30,110.5,18371.70
4,C10005,2023-03-05,MX,Basic,79,65.80,57.1,5198.20


In [11]:
# Calcular el churn, evitando división por cero o NaN
df['churn_risk'] = 1 - (df['last_usage_score'] / df['avg_monthly_usage_score'])
df['churn_risk'] = df['churn_risk'].replace([np.inf, -np.inf], np.nan)
df['churn_risk'] = df['churn_risk'].fillna(0)  # si no hay uso, podrías decidir otro criterio
df.head()

,customer_id,signup_date,country,plan_type,monthly_fee,avg_monthly_usage_score,last_usage_score,CLV,churn_risk
0,C10001,2023-01-14,MX,Plus,149,173.43,195.6,25841.57,-0.13
1,C10002,2022-11-03,CO,Basic,79,13.17,2.2,1040.17,0.83
2,C10003,2023-04-22,MX,Premium,199,304.70,325.3,60635.30,-0.07
3,C10004,2023-02-10,AR,Plus,149,123.30,110.5,18371.70,0.10
4,C10005,2023-03-05,MX,Basic,79,65.80,57.1,5198.20,0.13


In [12]:
# 8. Segmentar el riesgo
conditions = [
    df['churn_risk'] >= 0.7,
    (df['churn_risk'] >= 0.4) & (df['churn_risk'] < 0.7),
    df['churn_risk'] < 0.4
]

choices = ['High Risk', 'Medium Risk', 'Low Risk']

df['risk_segment'] = np.select(conditions, choices, default='Low Risk')
df.head()

,customer_id,signup_date,country,plan_type,monthly_fee,avg_monthly_usage_score,last_usage_score,CLV,churn_risk,risk_segment
0,C10001,2023-01-14,MX,Plus,149,173.43,195.6,25841.57,-0.13,Low Risk
1,C10002,2022-11-03,CO,Basic,79,13.17,2.2,1040.17,0.83,High Risk
2,C10003,2023-04-22,MX,Premium,199,304.70,325.3,60635.30,-0.07,Low Risk
3,C10004,2023-02-10,AR,Plus,149,123.30,110.5,18371.70,0.10,Low Risk
4,C10005,2023-03-05,MX,Basic,79,65.80,57.1,5198.20,0.13,Low Risk


In [13]:
# 9. Tabla final
final = df[['customer_id', 'CLV', 'churn_risk', 'risk_segment']]
final

,customer_id,CLV,churn_risk,risk_segment
0,C10001,25841.57,-1.28e-01,Low Risk
1,C10002,1040.17,8.33e-01,High Risk
2,C10003,60635.30,-6.76e-02,Low Risk
3,C10004,18371.70,1.04e-01,Low Risk
4,C10005,5198.20,1.32e-01,Low Risk
5,C10006,50526.10,2.64e-02,Low Risk
6,C10007,22573.50,-6.60e-03,Low Risk
7,C10008,666.23,6.68e-01,Medium Risk
8,C10009,44715.30,4.36e-02,Low Risk
9,C10010,15645.00,-9.43e-02,Low Risk


In [ ]:
# Opcional: exportar
# final.to_csv("customer_risk_clv.csv", index=False)

### Conclusión

El cálculo real por supuesto es más complejo debido a que los clientes se dan de baja y hay posteriores altas entre los meses del análisis, lo importante es tener en cuenta que este mismo código fue utilizado para realizar un score risk rápido para 5.5  millones de suscriptores, segmentando/categorizando fácilmente con base en métricas muy sencillas.